# Phases 3 et 4 — fine-tuning, puis cascade contre bout-en-bout

À exécuter de haut en bas ; chaque étape est sauvegardée sur Drive et peut
se reprendre dans une autre session.

1. **Whisper multi-tâche** sur Jeli-ASR (`--task both`) : un seul modèle,
   un seul encodeur, deux sorties. `<|transcribe|>` donne le bambara (la
   cascade), `<|translate|>` le français (le bout-en-bout).
2. **Évaluation de Whisper** sur le test, dès qu'il est affiné, sans
   attendre NLLB : de premiers résultats même si la session s'arrête là.
3. **NLLB**, bambara → français d'abord (le sens de la comparaison), sur
   Bayelemabaga.
4. **Évaluation complète**, cascade contre bout-en-bout, avec le protocole
   de la phase 1 (même partition, même graine, mêmes 300 énoncés).
5. **Test de l'assistant affiné**, jusqu'à la démo au micro.

**Durée.** Une époque de Whisper prend environ 2 h 50 sur un T4 (mesuré
le 6 octobre 2026, 3 781 étapes). Une session Colab gratuite peut être
coupée en route : les checkpoints sont sur Drive et `--resume` repart du
dernier. Après une coupure, relancer les sections 1 à 3, puis la cellule
interrompue. Les cellules d'entraînement ne refont rien si leur modèle est
déjà sur Drive.

*Exécution → Modifier le type d'exécution → GPU (T4).*

## 1. Installation

In [1]:
# Branche de travail : contient les dernières corrections du notebook et des
# scripts. Une fois fusionnée, la branche par défaut suffit (sans -b).
BRANCH = "claude/chat-ia-bambara-dho8oa"

!git clone -q -b $BRANCH https://github.com/LOSTHADY/IA_Project- projet
%cd projet
!pip install -q transformers datasets accelerate sentencepiece jiwer sacrebleu soundfile librosa

/content/projet
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 84.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.6/129.6 kB 13.3 MB/s eta 0:00:00


## 2. Dossiers sur Drive

In [2]:
from google.colab import drive
drive.mount("/content/drive")

CKPT = "/content/drive/MyDrive/bambara/modeles"            # checkpoints et modèles finaux
OUT = "/content/drive/MyDrive/bambara/resultats-phase3"    # rapports d'évaluation
!mkdir -p "$CKPT" "$OUT"

Mounted at /content/drive


## 3. Accès aux jeux et sous-ensemble Jeli-ASR

Même réglage qu'en phase 1. Si Jeli-ASR demande un sous-ensemble, reprendre
celui choisi alors : les chiffres ne se comparent que sur le même jeu.

In [3]:
JELI_CONFIG = ""  # ex. "bam-asr-oza", comme en phase 1

JELI = "--dataset RobotsMali/jeli-asr" + (f" --dataset-config {JELI_CONFIG}" if JELI_CONFIG else "")
JELI_EVAL = "--dataset RobotsMali/jeli-asr" + (f" --config {JELI_CONFIG}" if JELI_CONFIG else "")

# Si un jeu est « gated » : décommenter.
# from huggingface_hub import notebook_login; notebook_login()

# Époques d'entraînement. Une est un bon premier passage sur Colab gratuit ;
# l'essai du §4 annonce la durée de chaque option avant de lancer le long.
EPOCHS_WHISPER = 1
EPOCHS_NLLB = 1

In [4]:
# Modèle final s'il existe, sinon le dernier checkpoint (entraînement coupé
# ou inachevé), sinon le modèle d'origine quand il y en a un. Défini ici pour
# rester disponible après une reprise (sections 1 à 3) ; les sections 6 à 10
# appellent chemins().
from pathlib import Path
from transformers.trainer_utils import get_last_checkpoint

def modele(dossier, repli=None):
    if (Path(dossier) / "model.safetensors").exists():
        return dossier
    dernier = get_last_checkpoint(dossier) if Path(dossier).is_dir() else None
    if dernier:
        print(f"Entraînement inachevé : {dernier}")
        return dernier
    if repli:
        print(f"Pas encore entraîné ({dossier}) : modèle d'origine {repli}")
        return repli
    raise FileNotFoundError(f"ni modèle final ni checkpoint dans {dossier}")

def deja(label):
    """Vrai si un rapport porte déjà ce nom : relancer une cellule
    d'évaluation, d'une session à l'autre, ne la refait pas."""
    import glob, json
    for p in glob.glob(f"{OUT}/*.json"):
        if json.load(open(p)).get("nom") == label:
            print(f"« {label} » déjà évalué ({p}) : cellule sautée.")
            return True
    return False

def chemins():
    """Whisper, NLLB bambara → français et NLLB français → bambara à
    utiliser, d'après ce qui est sur Drive."""
    nllb = "facebook/nllb-200-distilled-600M"
    return (modele(f"{CKPT}/whisper-small-bm"),
            modele(f"{CKPT}/nllb-bm2fr", repli=nllb),
            modele(f"{CKPT}/nllb-fr2bm", repli=nllb))

## 4. Essai rapide (10 minutes) et durée prévue

400 enregistrements, une époque, soit une cinquantaine d'étapes. Vérifie
que tout s'enchaîne (données, colonnes, mémoire GPU) et mesure la vitesse
de ce GPU. Les scores de cet essai ne veulent rien dire.

In [5]:
!python scripts/finetune_whisper.py $JELI --model openai/whisper-small --task both \
    --max-samples 400 --epochs 1 --save-steps 50 --eval-samples 20 --output /content/essai

preprocessor_config.json: 100% 185k/185k [00:00<00:00, 270MB/s]
config.json: 100% 1.97k/1.97k [00:00<00:00, 6.89MB/s]
tokenizer_config.json: 100% 283k/283k [00:00<00:00, 346MB/s]
vocab.json: 100% 836k/836k [00:00<00:00, 107MB/s]
tokenizer.json: 100% 2.48M/2.48M [00:00<00:00, 165MB/s]
merges.txt: 100% 494k/494k [00:00<00:00, 61.6MB/s]
normalizer.json: 100% 52.7k/52.7k [00:00<00:00, 85.6MB/s]
added_tokens.json: 100% 34.6k/34.6k [00:00<00:00, 11.4MB/s]
special_tokens_map.json: 100% 2.19k/2.19k [00:00<00:00, 8.86MB/s]

model.safetensors: downloading bytes:  11% 111M/967M [00:01<00:09, 94.0MB/s, 8.78MB/s  ] 
model.safetensors: downloading bytes:  17% 167M/967M [00:02<00:08, 98.1MB/s, 14.2MB/s  ]
model.safetensors: downloading bytes:  20% 192M/967M [00:02<00:07, 105MB/s, 16.1MB/s  ] 
model.safetensors: reconstructing file:  25% 239M/967M [00:02<00:06, 111MB/s, 18.2MB/s  ] 
model.safetensors: downloading bytes:  24% 234M/967M [00:02<00:06, 119MB/s, 19.1MB/s  ]
model.safetensors: downloading b

In [6]:
# Durée prévue de l'entraînement complet, d'après la vitesse de l'essai.
import json, math
state = json.load(open("/content/essai/trainer_state.json"))  # écrit en fin d'essai
last = [h for h in state["log_history"] if "train_runtime" in h][-1]
s_per_step = last["train_runtime"] / state["global_step"]
steps_per_epoch = math.ceil(2 * 32_000 / (8 * 2))   # 2 tâches, lot 8 × accumulation 2
print(f"{s_per_step:.2f} s par étape (essai, démarrage compris : plutôt pessimiste)")
for epochs in (1, 2, 3):
    hours = epochs * steps_per_epoch * s_per_step * 1.1 / 3600   # +10 % d'évaluations
    print(f"Whisper, {epochs} époque(s) : environ {hours:.1f} h")

print("Réglage actuel (§3) :", EPOCHS_WHISPER, "époque(s) pour Whisper,", EPOCHS_NLLB, "pour NLLB")

3.51 s par étape (essai, démarrage compris : plutôt pessimiste)
Whisper, 1 époque(s) : environ 4.3 h
Whisper, 2 époque(s) : environ 8.6 h
Whisper, 3 époque(s) : environ 12.9 h
Réglage actuel (§3) : 1 époque(s) pour Whisper, 1 pour NLLB


## 5. Fine-tuning de Whisper (long)

Le meilleur checkpoint est choisi au WER sur une validation prise dans
`train`. La partition de test reste intacte pour l'évaluation (§7).

**Coupures.** Un checkpoint est écrit sur Drive toutes les 250 étapes
(environ un quart d'heure) : une coupure ne coûte que le travail depuis le
dernier. Après une coupure, relancer les sections 1 à 3 puis cette cellule,
qui reprend au dernier checkpoint. Si l'entraînement ne peut pas aller au
bout, le §7 évalue le dernier checkpoint.

In [7]:
from pathlib import Path
# model.safetensors n'est écrit qu'à la fin de l'entraînement. Sans ce
# garde-fou, une relance après le nettoyage du Drive repartirait de zéro et
# écraserait le modèle. (Pour réentraîner : supprimer ce dossier.)
if Path(f"{CKPT}/whisper-small-bm/model.safetensors").exists():
    print("Whisper déjà affiné : rien à faire.")
else:
    !python scripts/finetune_whisper.py $JELI --model openai/whisper-small --task both \
        --epochs $EPOCHS_WHISPER --save-steps 250 --eval-samples 150 \
        --output "$CKPT/whisper-small-bm" --resume

Loading weights: 100% 479/479 [00:00<00:00, 782.92it/s]
Schéma : {'audio': Audio(sampling_rate=16000, decode=True, num_channels=None, stream_index=None), 'duration': Value('float32'), 'bam': Value('string'), 'french': Value('string')}
Colonnes : bambara=bam, français=french, audio=audio
Entraînement : 60483 exemples (transcribe + translate) ; validation : 150 (transcribe, partition prise dans train)
Reprise depuis /content/drive/MyDrive/bambara/modeles/whisper-small-bm/checkpoint-3000
[transformers] There were missing keys in the checkpoint model loaded: ['proj_out.weight'].
[transformers] Warning: The following arguments do not match the ones in the `trainer_state.json` within the checkpoint directory: 
	eval_steps: 250 (from args) != 500 (from trainer_state.json)
	save_steps: 250 (from args) != 500 (from trainer_state.json)
{'loss': '2.505', 'grad_norm': '19.45', 'learning_rate': '8e-07', 'epoch': '0.8002'}
{'loss': '2.621', 'grad_norm': '25.76', 'learning_rate': '1.633e-06', 'epoch'

### Libérer le Drive

Un checkpoint de Whisper pèse environ 3 Go (poids et état de l'optimiseur),
et deux sont gardés. Une fois le modèle final écrit, ils ne servent plus :
le Drive gratuit (15 Go) en a besoin pour NLLB.

In [8]:
from pathlib import Path
import shutil
final = Path(f"{CKPT}/whisper-small-bm")
if (final / "model.safetensors").exists():
    for ck in sorted(final.glob("checkpoint-*")):
        shutil.rmtree(ck)
        print("supprimé :", ck.name)
else:
    print("Modèle final absent : checkpoints gardés pour reprendre l'entraînement.")
!du -sh "$CKPT"

supprimé : checkpoint-3000
supprimé : checkpoint-3781
927M	/content/drive/MyDrive/bambara/modeles


## 6. Évaluation de Whisper, sans attendre NLLB

Sur les 300 énoncés de test de la phase 1. Tant que NLLB n'est pas affiné,
la cellule suivante prend le NLLB d'origine : la cascade de cette section
associe donc Whisper affiné et NLLB d'origine. La cascade entièrement
affinée vient au §8.

**À m'envoyer** : les sorties du tableau, des tests de significativité et
de l'analyse d'erreurs, en fin de section.

In [9]:
WHISPER, NLLB_BM2FR, NLLB_FR2BM = chemins()

Pas encore entraîné (/content/drive/MyDrive/bambara/modeles/nllb-bm2fr) : modèle d'origine facebook/nllb-200-distilled-600M
Pas encore entraîné (/content/drive/MyDrive/bambara/modeles/nllb-fr2bm) : modèle d'origine facebook/nllb-200-distilled-600M


### Références non affinées, dans le même environnement

Les mêmes mesures qu'en phase 1 (`docs/RESULTATS.md`), recalculées ici sur
les mêmes énoncés, pour que les écarts avec les modèles affinés se testent
phrase par phrase. Une vingtaine de minutes sur GPU.

In [10]:
if not deja("MMS + NLLB zero-shot"):
    !python -m eval.baselines --out "$OUT" --label "MMS + NLLB zero-shot" \
        asr $JELI_EVAL --model facebook/mms-1b-all --kind ctc --target-lang bam --with-mt --limit 300
if not deja("NLLB zero-shot / Bayelemabaga"):
    !python -m eval.baselines --out "$OUT" --label "NLLB zero-shot / Bayelemabaga" \
        mt --dataset RobotsMaliAI/bayelemabaga --limit 500
if not deja("NLLB zero-shot / Jeli-ASR"):
    !python -m eval.baselines --out "$OUT" --label "NLLB zero-shot / Jeli-ASR" \
        mt $JELI_EVAL --limit 500

INFO NumExpr defaulting to 2 threads.
INFO TensorFlow version 2.21.0 available.
INFO JAX version 0.11.1 available.
INFO HTTP Request: HEAD https://huggingface.co/datasets/RobotsMali/jeli-asr/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
WARNING Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
INFO HTTP Request: HEAD https://huggingface.co/api/resolve-cache/datasets/RobotsMali/jeli-asr/e0afc02875ac1fb4601667c309fe351e00f06aaf/README.md?%2Fdatasets%2FRobotsMali%2Fjeli-asr%2Fresolve%2Fmain%2FREADME.md=&etag=%22974f6423d780dcafc665d3773f3526cf58a0a04d%22 "HTTP/1.1 200 OK"
INFO HTTP Request: HEAD https://huggingface.co/datasets/RobotsMali/jeli-asr/resolve/e0afc02875ac1fb4601667c309fe351e00f06aaf/jeli-asr.py "HTTP/1.1 404 Not Found"
INFO HTTP Request: HEAD https://s3.amazonaws.com/datasets.huggingface.co/datasets/datasets/RobotsMali/jeli-asr/RobotsMali/jeli-asr.py "HTTP/1.1 404 Not Found"
INFO

### Whisper affiné, dans les deux architectures

Le même modèle : `<|translate|>` pour le bout-en-bout, `<|transcribe|>`
puis NLLB pour la cascade.

In [11]:
if not deja("bout-en-bout affiné"):
    !python -m eval.baselines --out "$OUT" --label "bout-en-bout affiné" \
        asr $JELI_EVAL --model "$WHISPER" --task translate --limit 300

INFO NumExpr defaulting to 2 threads.
INFO TensorFlow version 2.21.0 available.
INFO JAX version 0.11.1 available.
INFO HTTP Request: HEAD https://huggingface.co/datasets/RobotsMali/jeli-asr/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
INFO HTTP Request: HEAD https://huggingface.co/api/resolve-cache/datasets/RobotsMali/jeli-asr/e0afc02875ac1fb4601667c309fe351e00f06aaf/README.md?%2Fdatasets%2FRobotsMali%2Fjeli-asr%2Fresolve%2Fmain%2FREADME.md=&etag=%22974f6423d780dcafc665d3773f3526cf58a0a04d%22 "HTTP/1.1 200 OK"
INFO HTTP Request: HEAD https://huggingface.co/datasets/RobotsMali/jeli-asr/resolve/e0afc02875ac1fb4601667c309fe351e00f06aaf/jeli-asr.py "HTTP/1.1 404 Not Found"
INFO HTTP Request: HEAD https://s3.amazonaws.com/datasets.huggingface.co/datasets/datasets/RobotsMali/jeli-asr/RobotsMali/jeli-asr.py "HTTP/1.1 404 Not Found"
INFO HTTP Request: GET https://huggingface.co/api/datasets/RobotsMali/jeli-asr/revision/e0afc02875ac1fb4601667c309fe351e00f06aaf "HTTP/1.1 200 OK"
INF

In [12]:
if not deja("cascade, NLLB d'origine"):
    !python -m eval.baselines --out "$OUT" --label "cascade, NLLB d'origine" \
        asr $JELI_EVAL --model "$WHISPER" --with-mt --limit 300

INFO NumExpr defaulting to 2 threads.
INFO TensorFlow version 2.21.0 available.
INFO JAX version 0.11.1 available.
INFO HTTP Request: HEAD https://huggingface.co/datasets/RobotsMali/jeli-asr/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
WARNING Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
INFO HTTP Request: HEAD https://huggingface.co/api/resolve-cache/datasets/RobotsMali/jeli-asr/e0afc02875ac1fb4601667c309fe351e00f06aaf/README.md?%2Fdatasets%2FRobotsMali%2Fjeli-asr%2Fresolve%2Fmain%2FREADME.md=&etag=%22974f6423d780dcafc665d3773f3526cf58a0a04d%22 "HTTP/1.1 200 OK"
INFO HTTP Request: HEAD https://huggingface.co/datasets/RobotsMali/jeli-asr/resolve/e0afc02875ac1fb4601667c309fe351e00f06aaf/jeli-asr.py "HTTP/1.1 404 Not Found"
INFO HTTP Request: HEAD https://s3.amazonaws.com/datasets.huggingface.co/datasets/datasets/RobotsMali/jeli-asr/RobotsMali/jeli-asr.py "HTTP/1.1 404 Not Found"
INFO

### Premier tableau

In [13]:
!python -m eval.compare "$OUT"/*.json

| Métrique | MMS + NLLB zero-shot | bout-en-bout affiné | NLLB zero-shot / Bayelemabaga | NLLB zero-shot / Jeli-ASR |
|---|---|---|---|---|
| Architecture | cascade | e2e | traduction | traduction |
| Items | 300 | 293 | 500 | 491 |
| WER strict | 58.2 % | — | — | — |
| WER relâché | 54.8 % | — | — | — |
| Écart orthographique | 3.4 % | — | — | — |
| CER strict | 25.7 % | — | — | — |
| chrF++ (depuis ASR) | 18.03 | 19.93 | — | — |
| chrF++ (texte de réf.) | 22.92 | — | — | — |
| Perte due à l'ASR | 4.89 | — | — | — |
| BLEU (depuis ASR) | 0.91 | 3.03 | — | — |
| chrF++ bm→fr | — | — | 21.80 | 22.59 |
| chrF++ fr→bm | — | — | 25.66 | 29.76 |
| chrF++ fr→bm (replié) | — | — | 32.49 | 32.16 |
| BLEU bm→fr | — | — | 4.84 | 3.26 |
| BLEU fr→bm | — | — | 7.25 | 8.26 |
| Sorties coupées bm→fr | — | — | 0.6 % | 0.8 % |
| Sorties coupées fr→bm | — | — | 1.2 % | 1.6 % |
| Latence totale (s) | 1.59 | 0.48 | 0.92 | 0.89 |
| RTF | 0.42 | 0.16 | — | — |


In [14]:
# Ce que Whisper affiné apporte à la reconnaissance (WER, mêmes énoncés que MMS)
!python -m eval.significance "$OUT"/asr-*.json --noms "MMS + NLLB zero-shot" "cascade, NLLB d'origine"
# Premier regard sur la comparaison centrale (la cascade n'a pas encore son NLLB affiné)
!python -m eval.significance "$OUT"/asr-*.json --noms "cascade, NLLB d'origine" "bout-en-bout affiné"

Traceback (most recent call last):
  File "<frozen runpy>", line 203, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/content/projet/eval/significance.py", line 338, in <module>
    main()
    ~~~~^^
  File "/content/projet/eval/significance.py", line 313, in main
    loaded = [load_series(path) for path in args.reports
              ~~~~~~~~~~~^^^^^^
  File "/content/projet/eval/significance.py", line 137, in load_series
    report = json.loads(report_path.read_text(encoding="utf-8"))
                        ~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.13/pathlib/_local.py", line 546, in read_text
    return PathBase.read_text(self, encoding, errors, newline)
           ~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.13/pathlib/_abc.py", line 632, in read_text
    with self.open(mode='r', encoding=encoding, errors=errors, newline=newline) as f:
         ~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^

In [15]:
!python -m eval.analysis "$OUT"/*.json | tee "$OUT/analyse-whisper.md"

### MMS + NLLB zero-shot

**ASR** (300 énoncés) — WER 58.2 % = substitutions 38.7 % + suppressions 10.3 % + insertions 9.2 %.

- Substitutions purement orthographiques : 7.0 %
- Sorties vides : 0.0 % ; hallucinations (sortie ≥ 2× la référence) : 2.0 % ; boucles de répétition : 0.0 %
- Longueur sortie / référence, médiane : 1.00
- WER selon la longueur de l'énoncé (mots) : 1–3 : 110.6 % (n=20) ; 4–7 : 56.0 % (n=103) ; 8–15 : 59.2 % (n=121) ; 16+ : 56.2 % (n=56)
- Confusions fréquentes : k'a → ka (11), bɛɛ → bɛ (8), bɛ → bi (6), n'i → ni (5), t'a → ta (4), kan → ka (4), do → don (4), ko → kɔ (4), k'o → ko (4), ye → ɲe (3)
- Mots les plus supprimés : ye (26), a (25), ne (15), n (13), i (10), o (9), la (9), na (8), de (7), ni (6)
- Mots les plus insérés : a (13), ka (10), la (9), ye (9), ni (7), de (6), ko (6), i (6), ma (6), e (6)

- *meilleur* (WER 0.0 %) — référence : « ɔwɔ ne bɛ yan » ; sortie : « ɔwɔ ne bɛ yan »
- *median* (WER 57.1 %) — référence : « a ba de ka nkɔyɔ foro don » ; sor

## 7. Fine-tuning de NLLB, un modèle par sens

**Place.** Un checkpoint de NLLB pèse environ 7,4 Go (poids et état de
l'optimiseur) : trop pour le Drive gratuit. Les checkpoints restent donc sur
le disque de Colab, et seul le modèle final (environ 2,5 Go) est copié sur
Drive. Contrepartie : une coupure pendant l'entraînement d'un sens oblige à
le reprendre de zéro.

**Ordre.** D'abord bambara → français : c'est le sens de la cascade, celui
de la comparaison avec le bout-en-bout. Français → bambara ne sert qu'aux
réponses, identiques dans les deux architectures : il peut attendre une
autre session.

In [16]:
import shutil
from pathlib import Path
if Path(f"{CKPT}/nllb-bm2fr/model.safetensors").exists():
    print("NLLB bm2fr déjà affiné : rien à faire.")
else:
    !python scripts/finetune_nllb.py --direction bm2fr --epochs $EPOCHS_NLLB --save-steps 500 \
        --output /content/nllb-bm2fr --resume
    # Modèle final seul vers Drive, sans les checkpoints.
    if Path("/content/nllb-bm2fr/model.safetensors").exists():
        shutil.copytree("/content/nllb-bm2fr", f"{CKPT}/nllb-bm2fr", dirs_exist_ok=True,
                        ignore=shutil.ignore_patterns("checkpoint-*"))
!du -sh "$CKPT"/*

Using the latest cached version of the dataset since RobotsMaliAI/bayelemabaga couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'default' at /root/.cache/huggingface/datasets/RobotsMaliAI___bayelemabaga/default/0.0.0/d3820304aa185094e11b18a75d8715218a3f54b1 (last modified on Wed Oct  7 16:37:32 2026).
Schéma : {'translation': Translation(languages=['bam', 'fr'])}
Colonnes : bambara=translation.bam, français=translation.fr, audio=—
Loading weights: 100% 512/512 [00:00<00:00, 27137.65it/s]
Map: 100% 75160/75160 [00:10<00:00, 7475.40 examples/s]
Map: 100% 300/300 [00:00<00:00, 5967.96 examples/s]
Aucun checkpoint : départ de zéro
  1% 37/4698 [00:21<43:15,  1.80it/s]Traceback (most recent call last):
  File "/content/projet/scripts/finetune_nllb.py", line 183, in <module>
    main()
    ~~~~^^
  File "/content/projet/scripts/finetune_nllb.py", line 171, in main
    trainer.train(resume_from_checkpoint=last)
    ~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^

In [ ]:
import shutil
from pathlib import Path
if Path(f"{CKPT}/nllb-fr2bm/model.safetensors").exists():
    print("NLLB fr2bm déjà affiné : rien à faire.")
else:
    !python scripts/finetune_nllb.py --direction fr2bm --epochs $EPOCHS_NLLB --save-steps 500 \
        --output /content/nllb-fr2bm --resume
    # Modèle final seul vers Drive, sans les checkpoints.
    if Path("/content/nllb-fr2bm/model.safetensors").exists():
        shutil.copytree("/content/nllb-fr2bm", f"{CKPT}/nllb-fr2bm", dirs_exist_ok=True,
                        ignore=shutil.ignore_patterns("checkpoint-*"))
!du -sh "$CKPT"/*

## 8. Évaluation complète — cascade contre bout-en-bout

Avec NLLB affiné. Relancer d'abord la cellule suivante : elle reprend les
modèles affinés depuis Drive.

In [ ]:
# Après le §7 : NLLB affiné s'il est sur Drive, sinon celui d'origine.
WHISPER, NLLB_BM2FR, NLLB_FR2BM = chemins()
MT_LABEL = None  # fixé par la cellule « traduction seule »

In [ ]:
if NLLB_BM2FR.startswith("facebook/"):
    print("NLLB bambara → français pas encore affiné (§7) : cellule sautée.")
elif not deja("cascade affinée"):
    !python -m eval.baselines --out "$OUT" --label "cascade affinée" \
        asr $JELI_EVAL --model "$WHISPER" --with-mt --mt-model "$NLLB_BM2FR" --limit 300

### Traduction seule, dans les deux sens

In [ ]:
# Libellé selon ce qui a été affiné : les deux sens, ou bambara → français seul.
if NLLB_BM2FR.startswith("facebook/"):
    MT_LABEL = None
    print("NLLB bambara → français pas encore affiné (§7) : cellule sautée.")
else:
    MT_LABEL = "NLLB affiné" if not NLLB_FR2BM.startswith("facebook/") else "NLLB bm→fr affiné"
    if not deja(f"{MT_LABEL} / Bayelemabaga"):
        !python -m eval.baselines --out "$OUT" --label "$MT_LABEL / Bayelemabaga" \
            mt --dataset RobotsMaliAI/bayelemabaga --limit 500 \
            --bm2fr-model "$NLLB_BM2FR" --fr2bm-model "$NLLB_FR2BM"
    if not deja(f"{MT_LABEL} / Jeli-ASR"):
        !python -m eval.baselines --out "$OUT" --label "$MT_LABEL / Jeli-ASR" \
            mt $JELI_EVAL --limit 500 \
            --bm2fr-model "$NLLB_BM2FR" --fr2bm-model "$NLLB_FR2BM"

### Tableau commun : références non affinées et modèles affinés

In [ ]:
!python -m eval.compare "$OUT"/*.json

### L'écart entre les architectures est-il réel ?

Intervalles de confiance à 95 % et test apparié par bootstrap, sur les mêmes
énoncés. Si l'IC de l'écart contient zéro, la différence n'est pas établie :
c'est ce qu'il faut écrire dans le mémoire, quelle que soit la taille
apparente de l'écart.

In [ ]:
# Ce que le fine-tuning apporte à la cascade (WER et chrF++, mêmes énoncés)
!python -m eval.significance "$OUT"/asr-*.json --noms "MMS + NLLB zero-shot" "cascade affinée"
# La comparaison centrale : cascade contre bout-en-bout (chrF++ depuis l'audio)
!python -m eval.significance "$OUT"/asr-*.json --noms "cascade affinée" "bout-en-bout affiné"
# Traduction seule, avant et après fine-tuning
if MT_LABEL:
    !python -m eval.significance "$OUT"/mt-*.json --noms "NLLB zero-shot / Bayelemabaga" "$MT_LABEL / Bayelemabaga"

### Comment les modèles affinés se trompent-ils ?

Même analyse d'erreurs que pour la phase 1 (`docs/RESULTATS.md`). À
vérifier en premier : le mot swahili « kwa » et les boucles de répétition,
qui expliquaient l'essentiel des insertions de Whisper zero-shot, doivent
avoir disparu. Le mot « kwa » trahirait un token de langue mal réaffecté
au bambara.

In [ ]:
!python -m eval.analysis "$OUT"/*.json | tee "$OUT/analyse.md"

## Lire les résultats

- **La comparaison centrale** est la ligne *chrF++ (depuis ASR)* : le
  français obtenu par la cascade (ASR puis NLLB) contre celui que Whisper
  produit directement, sur les mêmes énoncés et la même référence.
- **Propagation d'erreurs** (cascade) : ce que les erreurs d'ASR coûtent à
  la traduction. C'est le coût que le bout-en-bout cherche à supprimer.
  S'il reste faible, le maillon supplémentaire de la cascade est peu
  pénalisant, et elle garde ses avantages : texte bambara affichable et
  gabarits utilisables.
- **Latence** : mesurée sur GPU, donc indicative seulement. Le budget de
  déploiement se mesure sur CPU (phase 5).
- **Limites à écrire dans le mémoire** : Jeli-ASR est à la fois la source
  d'entraînement et d'évaluation. Si des locuteurs de test apparaissent
  aussi dans `train` (à vérifier dans la documentation du jeu), les scores
  sont optimistes. C'est pourquoi les chiffres définitifs se mesurent sur le
  jeu de test maison (phase 2), avec `eval.run_eval`.

## 9. Tester l'assistant affiné

La chaîne complète avec les modèles affinés : reconnaissance, traduction,
LLM (Qwen2.5-1.5B, ouvert), traduction, synthèse. D'abord sur 20 énoncés de
Jeli-ASR, dans les deux architectures, avec le temps de chaque étape et les
réponses produites ; puis en direct, au micro.

In [ ]:
WHISPER, NLLB_BM2FR, NLLB_FR2BM = chemins()
import json
CFG = "/content/assistant-affine.json"
json.dump({"device": "cuda",
           "asr": {"model_id": WHISPER},
           "mt_in": {"model_id": NLLB_BM2FR},
           "mt_out": {"model_id": NLLB_FR2BM}}, open(CFG, "w"), indent=1)
!python -m bambara_voice.cli --config $CFG check
!python scripts/export_corpus_sample.py $JELI_EVAL --limit 20 --out /content/echantillon

In [ ]:
!python -m eval.run_eval --testset /content/echantillon/testset.jsonl --arch cascade \
    --config $CFG --synthesize --out "$OUT/chaine" > /dev/null
!python -m eval.run_eval --testset /content/echantillon/testset.jsonl --arch e2e \
    --config $CFG --synthesize --out "$OUT/chaine" > /dev/null

import glob
for path in sorted(glob.glob(f"{OUT}/chaine/*-*[0-9].json")):
    r = json.load(open(path))
    rows = [json.loads(l) for l in open(path.replace(".json", ".details.jsonl"))]
    print(f"\n=== {r['architecture']} — {r['latence']['total_moyen_s']:.1f} s par tour :",
          r["latence"]["par_etape_s"])
    for x in rows[:8]:
        print(f"- dit : {x['ref_transcript_bm']}\n  compris : {x['source_bm'] or '—'}"
              f" | {x['source_fr']}\n  réponse : {x['reply_fr']}\n  en bambara : {x['reply_bm']}")

### Au micro

La cellule suivante affiche un lien `https://….gradio.live` : l'ouvrir, autoriser
le micro, parler en bambara. Elle tourne jusqu'à ce qu'on l'arrête (bouton
stop). Pour la variante bout-en-bout, ajouter `--arch e2e`.

In [ ]:
!pip install -q gradio
!python app/gradio_app.py --config $CFG --share

## 10. Préparer le déploiement sur CPU (phase 5)

Convertit les trois modèles en int8 (CTranslate2) sur Drive, avec le
fichier `config.json` qui les décrit. Télécharger ensuite le dossier
`modeles-cpu` sur la machine de déploiement : voir `docs/DEPLOIEMENT.md`
pour le LLM en GGUF, la perte de qualité à mesurer et la latence.

In [ ]:
WHISPER, NLLB_BM2FR, NLLB_FR2BM = chemins()
!pip install -q ctranslate2
!python scripts/export_cpu.py --out "$CKPT/../modeles-cpu" \
    --whisper "$WHISPER" \
    --nllb-bm2fr "$NLLB_BM2FR" --nllb-fr2bm "$NLLB_FR2BM"

## Facultatif — publier les modèles (en privé)

Pour les réutiliser ailleurs (démo, phase 5) sans passer par Drive.

In [ ]:
# from huggingface_hub import HfApi
# api = HfApi()
# for name in ("whisper-small-bm", "nllb-bm2fr", "nllb-fr2bm"):
#     repo = api.create_repo(f"VOTRE_COMPTE/{name}", private=True, exist_ok=True).repo_id
#     api.upload_folder(folder_path=f"{CKPT}/{name}", repo_id=repo,
#                       ignore_patterns=["checkpoint-*"])